In [0]:
from pyspark.sql.functions import *

In [0]:
facilities_df = spark.read.csv("abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/bronze/dbo.facilities.csv", header=True, inferSchema=True)

display(facilities_df)

In [0]:
facility_id = facilities_df.dropDuplicates(["facility_id"])
display(facility_id)

In [0]:
from pyspark.sql.functions import trim, initcap, upper, col

facility_name = facility_id.withColumn(
    "facility_name",
    upper(initcap(trim(col("facility_name"))))
)

display(facility_name)

In [0]:
from pyspark.sql.functions import upper

facility_type = facility_name.withColumn(
    "facility_type_std",
    upper(trim("facility_type"))
)
display(facility_type)

In [0]:
from pyspark.sql.functions import trim, upper

nabh_id_clean = facility_type.withColumn(
    "nabh_id_clean",
    upper(trim("nabh_id"))
)
display(nabh_id_clean)


In [0]:
from pyspark.sql.functions import when, col

is_nabh_accredited = nabh_id_clean.withColumn(
    "is_nabh_accredited",
    when(col("nabh_id").isNotNull(), 1).otherwise(0)
)
display(is_nabh_accredited)

In [0]:
from pyspark.sql.functions import col, when

total_beds_valid = is_nabh_accredited.withColumn(
    "total_beds_valid",
    when(col("total_beds") > 0, col("total_beds").cast("int"))
)
display(total_beds_valid)

In [0]:
icu_beds_valid = total_beds_valid.withColumn(
    "icu_beds_valid",
    when(
        (col("icu_beds") >= 0) & 
        (col("icu_beds") <= col("total_beds_valid")),
        col("icu_beds").cast("int")
    )
)
display(icu_beds_valid)

In [0]:
icu_beds_pct = icu_beds_valid.withColumn(
    "icu_beds_pct",
    when(
        col("total_beds_valid") > 0,
        (col("icu_beds_valid") / col("total_beds_valid")) * 100
    )
)
display(icu_beds_pct)

In [0]:
city_std = icu_beds_pct.withColumn(
    "city_std",
    initcap(trim("city"))
)
display(city_std)

In [0]:
state_std = city_std.withColumn(
    "state_std",
    upper(trim("state"))
)
display(state_std)

In [0]:
country_std = state_std.withColumn(
    "country_std",
    upper(trim("country"))
)
display(country_std)

In [0]:
phone_std = country_std.withColumn(
    "phone_std",
    when(
        col("contact_phone").isNull() | (col("contact_phone") == "N/A"),
        None
    ).otherwise(
        when(
            regexp_extract(col("contact_phone"), r"(\d{10})", 1) == "",
            None
        ).otherwise(
            regexp_extract(col("contact_phone"), r"(\d{10})", 1)
        )
    )
)
display(phone_std)



In [0]:
from pyspark.sql.functions import col, when, upper, trim

is_active_flag = phone_std.withColumn(
    "is_active_flag",
    when(col("is_active").isNull(), 1)  # NULL → 1
    .when(upper(trim(col("is_active"))).isin("Y", "YES", "1", "TRUE"), 1)
    .when(upper(trim(col("is_active"))).isin("N", "NO", "0", "FALSE"), 0)
    .otherwise(None)
)
display(is_active_flag)

In [0]:
from pyspark.sql.functions import col, when

hospital_size_category = is_active_flag.withColumn(
    "hospital_size_category",
    when(col("total_beds_valid").isNull(), "Unknown")
    .when(col("total_beds_valid") < 100, "Small")
    .when((col("total_beds_valid") >= 100) & (col("total_beds_valid") <= 500), "Medium")
    .when((col("total_beds_valid") >= 501) & (col("total_beds_valid") <= 1000), "Large")
    .when(col("total_beds_valid") > 1000, "Super Specialty")
)
display(hospital_size_category)

In [0]:
from pyspark.sql.functions import to_date, col

ingestion_date = hospital_size_category.withColumn(
    "ingestion_date",
    to_date(col("ingestion_timestamp"), "M/d/yy H:mm")
)
display(ingestion_date)

In [0]:
from pyspark.sql.functions import col, when

dq_facility_flag = ingestion_date.withColumn(
    "dq_facility_flag",
    when(
        col("facility_name").isNull() |
        col("city").isNull() |
        col("total_beds_valid").isNull(),
        1
    ).otherwise(0)
)
display(dq_facility_flag)

In [0]:

# ----------------------------------------
#  Write to Silver layer in delta format
# ----------------------------------------


TABLE_NAME = "facilities"  

silver_table_path = f"abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/silver/{TABLE_NAME}"

# ── OVERWRITE (Full Load) ────────────────────────────────────
dq_facility_flag.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_table_path)

print(f"✅ Data written to Silver layer: {silver_table_path}")